# 07 - Full Pipeline: End-to-End Lesson Generation
**Module:** `src/core/generator.py`

## What this notebook demonstrates
The complete Bantrly lesson generation pipeline -- from a teacher's
three inputs to a validated, saved, research-grounded lesson JSON.

## The full sequence
```
Teacher inputs: Grade Band + ELA Domain + Theme
        ↓
1. Pre-generation guardrail checks
        ↓
2. Prompt construction (system + few-shot + user)
        ↓
3. Groq API call (llama-3.3-70b-versatile)
        ↓
4. Output validation (strip fences → parse → check fields → guardrails)
        ↓
5. Save to data/generated/
        ↓
6. Register in deduplication registry
        ↓
Complete lesson JSON
```

## What we'll do
1. Generate a lesson and walk through every step
2. Inspect the generated lesson in full
3. Generate lessons across multiple grade bands
4. Demonstrate the deduplication registry growing
5. Show what happens when bad inputs are provided

In [1]:
import sys
sys.path.insert(0, '..')

from src.core.generator import LessonGenerator
from src.utils.file_handler import load_lesson, load_registry
from src.guardrails.checks import run_pre_checks

import json


## Part 1: Initialising the Generator
The generator loads your Groq API key from `.env` and
initialises the Groq client. If your key is missing or
invalid, it fails here -- not mid-generation.

In [2]:
gen = LessonGenerator(verbose=True)

[generator] LessonGenerator initialised ✅
[generator] Model: llama-3.3-70b-versatile


## Part 2: Previewing the Prompt
Before making any API call, let's see exactly what
the model will receive for our first generation request.
No tokens spent -- just inspection.

In [3]:
# Preview the prompt without making an API call
gen.preview_prompt(
    grade_band = "3-5",
    ela_domain = "Speaking",
    theme      = "Ocean Ecosystems"
)

Total messages: 4

[1] SYSTEM
----------------------------------------
You are an expert K-12 ELA lesson designer for Bantrly, a voice-based
language learning platform for students. Your job is to generate a single,
complete, structured lesson in JSON format.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LESSON DESIGN PRINCIPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Ev...

[2] USER
----------------------------------------
Here is a complete example of a well-designed Bantrly lesson for grade band 3-5. Study its structure, depth, and how it follows all the design principles:

{
  "lesson_id": "L-35-SPK-005",
  "metadata": {
    "grade_band": "3-5",
    "ela_domain": "Speaking",
    "lesson_type": "Mission Brief",
    ...

[3] ASSISTANT
----------------------------------------
Understood. I have studied the example lesson carefully. I will follow the same structure, depth, narrative quality, and JSON format exactly. I will respect all grade band rules and return only a valid JSON object 

## Part 3: Generating Our First Lesson
Now we make the real API call.
Printing the logs as it completes.

In [8]:
lesson = gen.generate(
    grade_band = "K-2",
    ela_domain = "Speaking",
    theme      = "Friendship and Kindness"
)


[generator] Starting lesson generation
[generator] Grade: K-2 | Domain: Speaking | Theme: Friendship and Kindness

[generator] Step 1 — Running pre-generation checks...
[checks] All pre-generation checks passed ✅
[generator] Step 2 — Checking deduplication...

[generator] Step 3 — Building prompt...
[generator] Prompt built — 10,582 chars (~2,645 tokens)

[generator] Step 4 — Calling Groq API (llama-3.3-70b-versatile)...
[generator] Attempt 1/3...
[generator] Received response — 2,860 chars

[generator] Step 5 — Validating output...
[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Describe a simple event using basic sequence words'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within K-2 vocabulary ceiling (30 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check pas

In [5]:
# High-level summary
print("=== LESSON SUMMARY ===\n")
print(f"ID             : {lesson['lesson_id']}")
print(f"Grade Band     : {lesson['metadata']['grade_band']}")
print(f"ELA Domain     : {lesson['metadata']['ela_domain']}")
print(f"Lesson Type    : {lesson['metadata']['lesson_type']}")
print(f"Theme          : {lesson['metadata']['theme']}")
print(f"Primary Skill  : {lesson['metadata']['primary_skill']}")
print(f"Voice Markers  : {lesson['metadata']['voice_markers']}")
print(f"Duration       : {lesson['metadata']['estimated_duration_minutes']} minutes")
print(f"CCSS Anchor    : {lesson['metadata']['ccss_anchor']}")

=== LESSON SUMMARY ===

ID             : L-912-RDG-010
Grade Band     : 9-12
ELA Domain     : Reading → Speaking
Lesson Type    : Text Explorer
Theme          : Persuasive Writing
Primary Skill  : Analyze the use of pathos, ethos, and logos in a persuasive text and explain how they contribute to the author's argument
Voice Markers  : ['Task Adherence', 'Prosody', 'Fluency & Fillers']
Duration       : 9 minutes
CCSS Anchor    : CCSS.ELA-Literacy.SL.11-12.3 — Evaluate a speaker's point of view, reasoning, and use of evidence and rhetoric, identifying any fallacious reasoning or exaggerated or distorted evidence.


## Part 4: Inspecting the Generated Lesson
Let's read the lesson carefully -- not just the metadata
but the actual content. This is the quality check.

In [9]:
# High-level summary
print("=== LESSON SUMMARY ===\n")
print(f"ID             : {lesson['lesson_id']}")
print(f"Grade Band     : {lesson['metadata']['grade_band']}")
print(f"ELA Domain     : {lesson['metadata']['ela_domain']}")
print(f"Lesson Type    : {lesson['metadata']['lesson_type']}")
print(f"Theme          : {lesson['metadata']['theme']}")
print(f"Primary Skill  : {lesson['metadata']['primary_skill']}")
print(f"Voice Markers  : {lesson['metadata']['voice_markers']}")
print(f"Duration       : {lesson['metadata']['estimated_duration_minutes']} minutes")
print(f"CCSS Anchor    : {lesson['metadata']['ccss_anchor']}")

=== LESSON SUMMARY ===

ID             : L-K2-SPK-001
Grade Band     : K-2
ELA Domain     : Speaking
Lesson Type    : Story Retell
Theme          : Friendship and Kindness
Primary Skill  : Describe a simple event using basic sequence words
Voice Markers  : ['Speaking Rate', 'Volume Control']
Duration       : 5 minutes
CCSS Anchor    : CCSS.ELA-Literacy.SL.K.4 — Describe familiar people, places, things, and events and, with prompting and support, provide additional detail.


In [10]:
# Read the full lesson flow
flow = lesson["lesson_flow"]

print("=== HOOK ===")
print(flow["hook"]["content"])

print("\n=== MODEL STAGE ===")
print(flow["model"]["skill_named_explicitly"])
print()
print(flow["model"]["content"])

print("\n=== PRACTICE PROMPTS ===")
for p in flow["practice"]:
    print(f"\n[{p['prompt_id']}] Type: {p['type']}")
    print(f"Prompt  : {p['text']}")
    if p.get("scaffold"):
        print(f"Scaffold: {p['scaffold']}")

print("\n=== REFLECT ===")
print(f"Voice marker focus : {flow['reflect']['voice_marker_focus']}")
print(f"Positive signal    : {flow['reflect']['positive_signal']}")
print(f"Growth signal      : {flow['reflect']['growth_signal']}")

=== HOOK ===
In a sunny meadow, two friends, Emma and Max, played together. They shared a big smile when they found a beautiful butterfly. But then, the butterfly flew away. Are you ready to hear what happened?

=== MODEL STAGE ===
Today we are practicing: describing a simple event using the words first and next.

Listen to how I describe what happened: 'First, Emma and Max played in the meadow. Next, they saw a beautiful butterfly.' Do you hear how those two words help the listener follow along? That's what we're practicing today.

=== PRACTICE PROMPTS ===

[P1] Type: supported
Prompt  : What did Emma and Max do first?
Scaffold: Start with: 'First, Emma and Max...'

[P2] Type: supported
Prompt  : What happened next with the butterfly?
Scaffold: Start with: 'Next, the butterfly...'

[P3] Type: independent
Prompt  : Tell me the whole story — what happened first and what happened next!

=== REFLECT ===
Voice marker focus : Volume Control
Positive signal    : A strong description sounds c

In [11]:
# Check the guardrail flags
print("=== GUARDRAIL FLAGS ===\n")
for check, result in lesson["guardrail_flags"].items():
    icon = "✅" if result["status"] == "pass" else "⚠️"
    print(f"{icon} {check}")
    print(f"   {result['message']}")
    print()

=== GUARDRAIL FLAGS ===

✅ single_skill_check
   Single skill confirmed: 'Describe a simple event using basic sequence words'

✅ vocabulary_ceiling_check
   All prompts within K-2 vocabulary ceiling (30 words).

✅ cognitive_load_check
   Cognitive load check passed for grade band 'K-2'.

✅ cultural_bias_check
   No cultural bias terms detected.



In [12]:
# Print the full lesson JSON
print("=== FULL LESSON JSON ===\n")
print(json.dumps(lesson, indent=2))

=== FULL LESSON JSON ===

{
  "lesson_id": "L-K2-SPK-001",
  "metadata": {
    "grade_band": "K-2",
    "ela_domain": "Speaking",
    "lesson_type": "Story Retell",
    "theme": "Friendship and Kindness",
    "primary_skill": "Describe a simple event using basic sequence words",
    "voice_markers": [
      "Speaking Rate",
      "Volume Control"
    ],
    "estimated_duration_minutes": 5,
    "ccss_anchor": "CCSS.ELA-Literacy.SL.K.4 \u2014 Describe familiar people, places, things, and events and, with prompting and support, provide additional detail.",
    "design_notes": "Single new skill only. Friendship theme keeps cognitive load low. Sequence words (first, next) are the scaffold \u2014 they reduce extraneous load while building germane load (Sweller, 1988)."
  },
  "lesson_flow": {
    "hook": {
      "duration_seconds": 60,
      "content": "In a sunny meadow, two friends, Emma and Max, played together. They shared a big smile when they found a beautiful butterfly. But then, the 

## Part 5: Generating Across Grade Bands
Let's generate one lesson per grade band to demonstrate
the system producing differentiated content automatically.

This is the range the brief asked for -- same pipeline,
fundamentally different outputs based on grade band rules.

In [13]:
# Generate a lesson for each grade band
# Using verbose=False to keep output clean

gen_quiet = LessonGenerator(verbose=False)

grade_requests = [
    ("K-2",  "Speaking",           "Nature & Animals"),
    ("6-8",  "Speaking",           "Climate Change"),
    ("9-12", "Reading → Speaking", "Artificial Intelligence Ethics"),
]

generated_lessons = []

for grade, domain, theme in grade_requests:
    print(f"Generating: {grade} | {domain} | {theme}...")
    l = gen_quiet.generate(grade_band=grade, ela_domain=domain, theme=theme)
    generated_lessons.append(l)
    print(f"  ✅ {l['lesson_id']} — {l['metadata']['primary_skill'][:60]}")
    print()

Generating: K-2 | Speaking | Nature & Animals...
[checks] All pre-generation checks passed ✅
[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ⚠️ [FLAG] primary_skill may describe more than one skill (contains 'and'): 'Speak audibly and express thoughts clearly about a familiar topic'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within K-2 vocabulary ceiling (30 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band 'K-2'.
[checks] cultural_bias_check: ✅ [PASS] No cultural bias terms detected.
[validator] Step 4 — Guardrail checks complete ✅
[validator] Validation passed for lesson: L-K2-SPK-002
[file_handler] Saved lesson → D:\projects\bantrly-lesson-generator\data\generated\L-K2-SPK-002.json
[file_handler] Registered combo → K-2 | Nature & Animals | Speak audibly and express tho

In [14]:
# Compare the generated lessons side by side
print("=== CROSS GRADE BAND COMPARISON ===\n")
print(f"{'Grade':<8} {'Type':<15} {'Skill':<55} {'Scaffolded?'}")
print(f"{'-'*8:<8} {'-'*15:<15} {'-'*55:<55} {'-'*11}")

for l in generated_lessons:
    grade    = l["metadata"]["grade_band"]
    ltype    = l["metadata"]["lesson_type"]
    skill    = l["metadata"]["primary_skill"][:52] + "..."
    practice = l["lesson_flow"]["practice"]
    has_scaffold = any(p.get("scaffold") for p in practice)
    print(f"{grade:<8} {ltype:<15} {skill:<55} {'Yes' if has_scaffold else 'No'}")

=== CROSS GRADE BAND COMPARISON ===

Grade    Type            Skill                                                   Scaffolded?
-------- --------------- ------------------------------------------------------- -----------
K-2      Mission Brief   Speak audibly and express thoughts clearly about a f... Yes
6-8      Mission Brief   Present a claim with at least two pieces of supporti... No
9-12     Text Explorer   Evaluate an author's use of evidence to support a cl... No


## Part 6: The Deduplication Registry
Every generation logs its combination to the registry.
Let's see how it's grown across our generations.

In [15]:
registry = load_registry()
combos   = registry["used_combinations"]

print(f"Registry now contains {len(combos)} entries:\n")
for entry in combos:
    print(f"  [{entry['grade_band']}] {entry['theme']}")
    print(f"         Skill     : {entry['skill'][:60]}...")
    print(f"         Lesson ID : {entry['lesson_id']}")
    print(f"         Logged at : {entry.get('generated_at', 'N/A')}")
    print()

Registry now contains 13 entries:

  [3-5] politics
         Skill     : Explain a process step by step...
         Lesson ID : L-NB-TEST-002
         Logged at : 2026-03-12T17:34:02.094052

  [3-5] Ocean Ecosystems
         Skill     : Explain a process step by step...
         Lesson ID : L-NB-TEST-002
         Logged at : 2026-03-12T17:34:15.994680

  [3-5] Ocean Ecosystems
         Skill     : Describe a habitat using sensory details and basic needs...
         Lesson ID : L-35-SPK-009
         Logged at : 2026-03-12T19:54:01.502037

  [9-12] Climate Change
         Skill     : Analyze an author's use of evidence to support a claim about...
         Lesson ID : L-912-RDG-SPK-009
         Logged at : 2026-03-12T19:55:22.297580

  [9-12] Persuasive Writing
         Skill     : Analyze and explain the effectiveness of persuasive techniqu...
         Lesson ID : L-912-RDG-SPK-009
         Logged at : 2026-03-12T19:55:53.660668

  [9-12] Persuasive Writing
         Skill     : Analyze a

## Part 7: Guardrail Demonstration
Let's show the guardrails catching bad inputs
before any API call is made.

In [16]:
# Bad grade band
print("=== BAD GRADE BAND ===\n")
try:
    gen.generate(
        grade_band = "Year 5",
        ela_domain = "Speaking",
        theme      = "Animals"
    )
except ValueError as e:
    print("Caught ValueError ✅")
    print(e)

=== BAD GRADE BAND ===


[generator] Starting lesson generation
[generator] Grade: Year 5 | Domain: Speaking | Theme: Animals

[generator] Step 1 — Running pre-generation checks...
Caught ValueError ✅
[checks] Pre-generation checks failed:
  ⚠️ [FLAG] 'Year 5' is not a valid grade band. Must be one of: ['3-5', '6-8', '9-12', 'K-2']
  ⚠️ [FLAG] Theme 'Animals' is too short. Please provide a descriptive theme.


In [17]:
# Bad ELA domain
print("=== BAD ELA DOMAIN ===\n")
try:
    gen.generate(
        grade_band = "3-5",
        ela_domain = "Maths",
        theme      = "Animals"
    )
except ValueError as e:
    print("Caught ValueError ✅")
    print(e)

=== BAD ELA DOMAIN ===


[generator] Starting lesson generation
[generator] Grade: 3-5 | Domain: Maths | Theme: Animals

[generator] Step 1 — Running pre-generation checks...
Caught ValueError ✅
[checks] Pre-generation checks failed:
  ⚠️ [FLAG] 'Maths' is not a valid ELA domain. Must be one of: ['Listening', 'Reading', 'Reading → Speaking', 'Speaking', 'Writing']
  ⚠️ [FLAG] Theme 'Animals' is too short. Please provide a descriptive theme.


In [18]:
# Empty theme
print("=== EMPTY THEME ===\n")
try:
    gen.generate(
        grade_band = "3-5",
        ela_domain = "Speaking",
        theme      = ""
    )
except ValueError as e:
    print("Caught ValueError ✅")
    print(e)

=== EMPTY THEME ===


[generator] Starting lesson generation
[generator] Grade: 3-5 | Domain: Speaking | Theme: 

[generator] Step 1 — Running pre-generation checks...
Caught ValueError ✅
[checks] Pre-generation checks failed:
  ⚠️ [FLAG] Theme cannot be empty.


## Part 8: Loading a Previously Generated Lesson
Every lesson is saved to disk. We can reload any lesson
by its ID -- the system has persistent memory across sessions.

In [19]:
# Load the first lesson we generated in this notebook
first_id = generated_lessons[0]["lesson_id"]
print(f"Loading lesson {first_id} from disk...\n")

reloaded = load_lesson(first_id)

print(f"lesson_id      : {reloaded['lesson_id']}")
print(f"grade_band     : {reloaded['metadata']['grade_band']}")
print(f"primary_skill  : {reloaded['metadata']['primary_skill']}")
print(f"theme          : {reloaded['metadata']['theme']}")
print()
print("Round-trip confirmed ✅ — lesson survived save and reload.")

Loading lesson L-K2-SPK-002 from disk...

lesson_id      : L-K2-SPK-002
grade_band     : K-2
primary_skill  : Speak audibly and express thoughts clearly about a familiar topic
theme          : Nature & Animals

Round-trip confirmed ✅ — lesson survived save and reload.


## Summary: What This Pipeline Demonstrates

| Stage | Module | What it proved |
|---|---|---|
| Pre-checks | `checks.py` | Bad inputs stopped before any API call |
| Prompt construction | `templates.py` | Grade-band rules injected dynamically |
| Groq API call | `generator.py` | Free-tier LLM produces structured lessons |
| Validation | `validator.py` | Malformed output caught and corrected |
| Persistence | `file_handler.py` | Lessons saved and reloaded reliably |
| Deduplication | `file_handler.py` | Registry grows with every generation |

## Research connections visible in the output
- **Narrative frame** (Bruner, 1990) — every lesson has a story hook
- **Worked example** (Sweller, 1994) — model stage shows skill before asking for it
- **Gradual release** (Vygotsky) — practice moves supported → independent
- **Feedback loop** (Hattie & Timperley, 2007) — reflect stage is specific, not generic
- **Grade differentiation** (Sweller, 1988) — K-2 and 9-12 lessons are fundamentally different

## Known limitations
- The `single_skill_check` produces false positives when "and" appears
  in descriptive language rather than joining two separate skills
- Cultural bias detection is keyword-based — misses subtle bias
- One few-shot example per grade band — true RAG would use embeddings
- No structured output mode (Groq supports `response_format`) —
  would eliminate the practice dict bug entirely in production